In [1]:
!pip install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 10.6 MB/s eta 0:00:00


In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [3]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

print("YOLO26 loaded successfully!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
YOLO26 loaded successfully!


In [6]:
from ultralytics import YOLO

model = YOLO("yolo26m.pt")

print("YOLO26 loaded successfully!")

YOLO26 loaded successfully!


In [5]:
from ultralytics import YOLO

model = YOLO("yolo26l.pt")

print("YOLO26 loaded successfully!")

YOLO26 loaded successfully!


In [3]:
from ultralytics import YOLO

model = YOLO("yolo26x.pt")

print("YOLO26 loaded successfully!")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
YOLO26 loaded successfully!


In [4]:
!git clone https://github.com/talhashafi-04/movx.git

Cloning into 'movx'...
remote: Enumerating objects: 2703, done.
remote: Total 2703 (delta 0), reused 0 (delta 0), pack-reused 2703 (from 2)
Receiving objects: 100% (2703/2703), 1.95 GiB | 33.20 MiB/s, done.
Updating files: 100% (2422/2422), done.


In [5]:
import os
from pathlib import Path

print("cwd:", os.getcwd())
root = Path("/content/movx/trimmed")
print("exists:", root.exists())
print("clips found:", len(list(root.glob("p*/*.mp4"))))

cwd: /content
exists: True
clips found: 84


In [7]:
import time
from ultralytics import YOLO

model = YOLO("yolo26x-pose.pt")
clip = "/content/movx/trimmed/p1/20261008_120913_trim.mp4"

t = time.time()
n = sum(1 for _ in model.predict(clip, imgsz=1280, stream=True, verbose=False))
print(n, "frames in", round(time.time() - t, 1), "s")

265 frames in 64.6 s


In [8]:
from google.colab import drive
drive.mount("/content/drive")

import time
from pathlib import Path
import numpy as np
from ultralytics import YOLO

model = YOLO("yolo26x-pose.pt")
ROOT = Path("/content/movx")
out_dir = Path("/content/drive/MyDrive/movx_poses/yolo26x")
out_dir.mkdir(parents=True, exist_ok=True)
MAX_PEOPLE = 3

clips = sorted((ROOT / "trimmed").glob("p*/*.mp4"))
for n, clip in enumerate(clips, 1):
    out = out_dir / f"{clip.parent.name}_{clip.stem}.npz"
    if out.exists():
        continue

    t = time.time()
    kpts, boxes = [], []
    for r in model.predict(str(clip), imgsz=1280, device=0, half=True, stream=True, verbose=False):
        k = np.full((MAX_PEOPLE, 17, 3), np.nan)
        b = np.full((MAX_PEOPLE, 4), np.nan)
        if r.boxes is not None and len(r.boxes) > 0:
            xyxy = r.boxes.xyxy.cpu().numpy()
            xy = r.keypoints.xy.cpu().numpy()
            conf = r.keypoints.conf.cpu().numpy()
            order = np.argsort(-(xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1]))
            for slot, i in enumerate(order[:MAX_PEOPLE]):
                k[slot, :, :2] = xy[i]
                k[slot, :, 2] = conf[i]
                b[slot] = xyxy[i]
        kpts.append(k)
        boxes.append(b)

    np.savez_compressed(out, kpts=np.array(kpts), boxes=np.array(boxes))
    print(f"{n}/{len(clips)} {clip.parent.name}/{clip.name}: {len(kpts)} frames in {time.time()-t:.0f}s", flush=True)

Mounted at /content/drive
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
1/84 p1/20261008_120553_trim.mp4: 264 frames in 22s
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
2/84 p1/20261008_120613_trim.mp4: 263 frames in 20s
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
3/84 p1/20261008_120638_trim.mp4: 266 frames in 19s
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
4/84 p1/20261008_120659_trim.mp4: 262 frames in 19s
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
5/84 p1/20261008_120749_trim.mp4: 263 frames in 19s
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
6/84 p1/20261008_120806_trim.mp4: 265 frames in 19s
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
7/84 p1/20261008_120821_t

In [10]:
!pip install -q transformers

In [11]:
import numpy as np, cv2, torch
from pathlib import Path
from PIL import Image
from transformers import AutoProcessor, VitPoseForPoseEstimation

ROOT = Path("/content/movx")
YOLO_DIR = Path("/content/drive/MyDrive/movx_poses/yolo26x")
OUT_DIR = Path("/content/drive/MyDrive/movx_poses/vitpose_plus_base")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NAME = "usyd-community/vitpose-plus-base"     # try ...-plus-huge later
processor = AutoProcessor.from_pretrained(NAME)
model = VitPoseForPoseEstimation.from_pretrained(NAME, device_map="auto").eval()

def run_clip(clip, batch_size=16):
    out = OUT_DIR / f"{clip.parent.name}_{clip.stem}.npz"
    boxes = np.load(YOLO_DIR / out.name)["boxes"][:, 0]   # largest person per frame, xyxy
    cap = cv2.VideoCapture(str(clip))
    T = len(boxes)
    kpts = np.full((T, 17, 2), np.nan)
    conf = np.full((T, 17), np.nan)

    def flush(images, idx, bxs):
        inputs = processor(images, boxes=bxs, return_tensors="pt").to(model.device)
        ds = torch.zeros(len(images), dtype=torch.long, device=model.device)   # 0 = COCO head
        with torch.no_grad():
            res = model(**inputs, dataset_index=ds)
        res = processor.post_process_pose_estimation(res, boxes=bxs)
        for i, people in zip(idx, res):
            kpts[i] = people[0]["keypoints"].cpu().numpy()
            conf[i] = people[0]["scores"].cpu().numpy()

    images, idx, bxs = [], [], []
    for t in range(T):
        ok, frame = cap.read()
        if not ok:
            break
        b = boxes[t]
        if np.isnan(b).any():
            continue                                  # no person found in this frame
        images.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        idx.append(t)
        x1, y1, x2, y2 = b
        bxs.append(np.array([[x1, y1, x2 - x1, y2 - y1]]))   # xyxy -> xywh
        if len(images) == batch_size:
            flush(images, idx, bxs)
            images, idx, bxs = [], [], []
    if images:
        flush(images, idx, bxs)
    cap.release()
    np.savez_compressed(out, kpts=kpts, conf=conf)
    return out

import time
t = time.time()
clip = ROOT / "trimmed/p1/20261008_120913_trim.mp4"
print(run_clip(clip), round(time.time() - t, 1), "s")

preprocessor_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.82k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  502MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

/content/drive/MyDrive/movx_poses/vitpose_plus_base/p1_20261008_120913_trim.npz 19.1 s


In [12]:
clips = sorted((ROOT / "trimmed").glob("p*/*.mp4"))
for n, clip in enumerate(clips, 1):
    if (OUT_DIR / f"{clip.parent.name}_{clip.stem}.npz").exists():
        continue
    t = time.time()
    run_clip(clip)
    print(f"{n}/{len(clips)} {clip.parent.name}/{clip.name} {time.time()-t:.0f}s", flush=True)

1/84 p1/20261008_120553_trim.mp4 30s
2/84 p1/20261008_120613_trim.mp4 24s
3/84 p1/20261008_120638_trim.mp4 15s
4/84 p1/20261008_120659_trim.mp4 15s
5/84 p1/20261008_120749_trim.mp4 15s
6/84 p1/20261008_120806_trim.mp4 17s
7/84 p1/20261008_120821_trim.mp4 15s
8/84 p1/20261008_120836_trim.mp4 17s
9/84 p1/20261008_120856_trim.mp4 24s
11/84 p1/20261008_120932_trim.mp4 15s
12/84 p1/20261008_120949_trim.mp4 15s
13/84 p10/20261008_140933_trim.mp4 12s
14/84 p10/20261008_141035_trim.mp4 8s
15/84 p10/20261008_141055_trim.mp4 14s
16/84 p10/20261008_141140_trim.mp4 10s
17/84 p10/20261008_141221_trim.mp4 13s
18/84 p10/20261008_141320_trim.mp4 14s
19/84 p10/20261008_141837_trim.mp4 16s
20/84 p10/20261008_142006_trim.mp4 16s
21/84 p11/20261008_141432_trim.mp4 14s
22/84 p11/20261008_141443_trim.mp4 16s
23/84 p11/20261008_141512_trim.mp4 15s
24/84 p12/20261008_143347_trim.mp4 15s
25/84 p12/20261008_143509_trim.mp4 15s
26/84 p12/20261008_143532_trim.mp4 15s
27/84 p13/20261008_143547_trim.mp4 15s
28/84 p